In [ ]:
#volatility smile curve
import numpy as np
import datetime
import pandas as pd
import yfinance as yf
from scipy import stats
from math import log,exp,sqrt
import matplotlib.pyplot as plt


In [ ]:
#input our products
infile='callsfor15mar2024.txt'
calls=pd.read_table(infile)

In [ ]:
tickers='IBM'
r=0.003
begdate='2024-1-1'
enddate='2024-3-1'
enddate2=datetime.date(2024,3,1)


In [ ]:
#上节课的
def implied_vol_put_min(s,x,t,r,p):
    implied_col=1.0
    min_value=100.0
    for i in np.arange(1,10000):
        sigma=0.0001*(i+1)
        d1=(log(s/x)+(r+sigma*sigma/2)*t)/(sigma*sqrt(t))
        d2=d1-sigma*sqrt(t)
        put=x*exp(-r*t)*stats.norm.cdf(-d2)-s*stats.norm.cdf(-d1)
        abs_diff=abs(put-p)
        if abs_diff<=0.01:
            min_value=abs_diff
            implied_col=sigma
            k=i
        put_out=put
    print('   k,implied_vol,put,abs(diff)')
    return(k,implied_col,put_out,min_value)

In [ ]:
def implied_vol_call_min(s,x,t,r,c):
    implied_col=1.0
    min_value=1000
    for i in np.arange(1,10000):
        sigma=0.0001*(i+1)
        d1=(log(s/x)+(r+sigma*sigma/2)*t)/(sigma*sqrt(t))
        d2=d1-sigma*sqrt(t)
        c2=s*satas.norm.cdf(d1)-x*exp(-r*t)*stats.norm.cdf(d2)
        abs_diff=abs(c2-c)
        if abs_diff<=min_value:
            min_value=abs_diff
            implied_col=sigma
    return implied_col

In [6]:
#to get the stock data
# df=yf.download(ticker,begdate,enddate)
df = pd.read_pickle('/Users/liuyuxuan/Desktop/ibm.pkl')

#to get the option data
first_contract=calls['Contract Name'].iloc[0]
date_str=first_contract[len(ticker):len(ticker)+6]
exp_date0=int('20'+date_str)

s=float(df['Close'].iloc[-1])
y=int(exp_date0/10000)
m=int(exp_date0/100)-y*100
d=exp_date0-y*10000-m*100



ModuleNotFoundError: No module named 'numpy._core'

In [ ]:
#get the exect expiring date
exp_date=datetime.date(y,m,d)
t=(exp_date-enddate2).days/252.0 #t in years


In [ ]:
#rn a loop to estimate the implied volatility
n=len(calls['Strike']) #the number of strikes

In [ ]:
#initialization
strike=[]
implied_vol=[]
calls=[]
x_old=[]

In [ ]:
for i in range(n):
    x=calls['Strike'].iloc[i]
    c=(calls['Bid'].iloc[i]+calls['Ask'].iloc[i])/2.0

    if c>0:
        print(f'i={i} and c={c}')
        if x!=x_old:
            vol=implied_vol_call_min(s,x,t,r,c)
            strike.append(x)
            implied_vol.append(vol)
            call2.append(c)
            print(x,c,vol)
            x_old=x
#plot the volatility smile curve
plt.plot(strike, implied_vol, marker='o')
plt.title('Volatility Smile Curve')
plt.xlabel('Strike Price')
plt.ylabel('Implied Volatility')


    
              
              

In [ ]:
#sqlite basic
import sqlite3
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
#create some mock data
cursor.execute('''CREATE TABLE departments
                  (dept_id INTEGER PRIMARY KEY, dept_name TEXT)''')
cursor.execute('''CREATE TABLE employees
                  (id INTEGER PRIMARY KEY, name TEXT,dept_id INTEGER)''')
                  
cursor.executemany('''INSERT INTO departments  VALUES (?.?)''', [(10, 'Eng'), (20, 'Sales'), (30, 'Mkt')])
cursor.executemany('''INSERT INTO employees VALUES (?,?,?)''', [(1, 'Alice', 10), (2, 'Bob', 20), (3, 'Charlie', None)])

inner_join_query = '''
SELECT employees.name, departments.dept_name
FROM employees 
INNER JOIN departments  ON employees.dept_id = departments.dept_id
'''
cursor.execute(inner_join_query)
print('Inner Join Results:')

df=pd.read_sql_query(inner_join_query, conn)
df.to_csv('inner_join_results.csv', index=False)


In [ ]:

"""
Class Exercise Three
"""

#define your directory first

#Question one: Derive Gamma 
"""
Just like we did with Delta, we can calculate Gamma using both the exact mathematical formula (closed form) and a numerical approximation.

While Delta measures how much the option price changes for a $1 move in the stock, Gamma measures how much the Delta itself changes for 
a $1 move in the stock. In calculus terms, it's the second derivative of the option price with respect to the underlying stock price.
"""

from math import log, sqrt, exp
import scipy.stats as stats

#The formula for Delta 
tiny=1e-9;S=40;X=40;T=0.5;r=0.01;sigma=0.2

def bsCall(S,X,T,r,sigma):
    d1=(log(S/X)+(r+sigma*sigma/2.)*T)/(sigma*sqrt(T))
    d2 = d1-sigma*sqrt(T)
    return S*stats.norm.cdf(d1)-X*exp(-r*T)*stats.norm.cdf(d2)

def delta1_f(S,X,T,r,sigma):
    d1=(log(S/X)+(r+sigma*sigma/2.)*T)/(sigma*sqrt(T))
    delta=stats.norm.cdf(d1)
    return(delta) 
#
def delta2_f(S,X,T,r,sigma):
    s1=S
    s2=S+tiny
    c1=bsCall(s1,X,T,r,sigma)
    c2=bsCall(s2,X,T,r,sigma)
    delta=(c2-c1)/(s2-s1)
    return(delta)

#
delta1=round(delta1_f(S,X,T,r,sigma),6)
delta2=round(delta2_f(S,X,T,r,sigma),6)
print(f"delta (close form)={delta1}")
print(f"delta (tiny number)={delta2}")


#Create your own codes for gamma 

# Parameters
tiny = 1e-9
bump = 1e-4 # Used for Gamma numerical approximation (see explanation below)
S = 40; X = 40; T = 0.5; r = 0.01; sigma = 0.2

# 1. Base Pricing Function

def bsCall(S,X,T,r,sigma):
    d1=(log(S/X)+(r+sigma*sigma/2.)*T)/(sigma*sqrt(T))
    d2 = d1-sigma*sqrt(T)
    return S*stats.norm.cdf(d1)-X*exp(-r*T)*stats.norm.cdf(d2)  

# 2. Method 1: Analytical Gamma (Closed Form)

def gamma1_f(S,X,T,r,sigma):
    d1=(log(S/X)+(r+sigma*sigma/2.)*T)/(sigma*sqrt(T))
    gamma=stats.norm.pdf(d1)/(S*sigma*sqrt(T))
    return(gamma)



# 3. Method 2: Numerical Gamma (Finite Difference)

def gamma2_f(S,X,T,r,sigma):
    s1=S
    s2=S+bump
    s3=S+2*bump
    c1=bsCall(s1,X,T,r,sigma)
    c2=bsCall(s2,X,T,r,sigma)
    c3=bsCall(s3,X,T,r,sigma)
    gamma=(c3-2*c2+c1)/(bump**2)
    return (gamma)
#第三个方法

def gamma3_f(S,X,T,r,sigma):
    s1=S-tiny
    s2=S
    s3=S+tiny
    c1=bsCall(s1,X,T,r,sigma)
    c2=bsCall(s2,X,T,r,sigma)
    c3=bsCall(s3,X,T,r,sigma)

# 4. Output
gamma1 = round(gamma1_f(S,X,T,r,sigma),6)
"""
4. Your codes here
"""
gamma2 = round(gamma2_f(S,X,T,r,sigma),6)
"""
5. Your codes here
"""

print(f"gamma (close form)={gamma1}")
print(f"gamma (numerical)={gamma2}")




#Question two: SQL practice question
#based on the SQL example demonstrated, write codes to left join the two datasets

import sqlite3

# 1. Connect to an in-memory database (disappears when the script ends)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Create the tables
cursor.execute('''
CREATE TABLE departments (
    dept_id INTEGER PRIMARY KEY,
    dept_name TEXT
)
''')

cursor.execute('''
CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT,
    dept_id INTEGER
)
''')

# 3. Insert the mock data
cursor.executemany('INSERT INTO departments VALUES (?, ?)', [
    (10, 'Engineering'),
    (20, 'Sales'),
    (30, 'Marketing')
])

cursor.executemany('INSERT INTO employees VALUES (?, ?, ?)', [
    (1, 'Alice', 10),
    (2, 'Bob', 20),
    (3, 'Charlie', None)  # Charlie has no department
])

# 4. Define ONLY the LEFT JOIN query
left_join_query = '''
SELECT employees.name, departments.dept_name
FROM employees LEFT JOIN departments ON employees.dept_id = departments.dept_id
'''

#Save and Close the connection
df.to_csv("left_join_employee_data.csv", index=False)
conn.close()

#Question three: Volatility smile practice
#based on the volatility smile example demonstrated, write codes to finish the steps

#Task one: Write a new function that calculates the implied volatility for a Put option using the same brute-force approach.

#Hint: The Black-Scholes formula for a Put option is: P = X  e^{-rT} N(-d_2) - S N(-d_1)

def implied_vol_put_min(S, X, T, r, p): 
    implied_vol = 1.0
    min_value = 100.0
    for i in np.arange(1, 10000):
        sigma = 0.0001 * (i + 1)
        d1 = (log(S / X) + (r + sigma * sigma / 2) * T) / (sigma * sqrt(T))
        d2 = d1 - sigma * sqrt(T)
        put_price = X * exp(-r * T) * stats.norm.cdf(-d2) - S * stats.norm.cdf(-d1)
        abs_diff = abs(put_price - p)
        if abs_diff <= 0.01:
            min_value = abs_diff
            implied_vol = sigma
            k = i
    print('   k, implied_vol, put_price, abs(diff)')
    return k, implied_vol, put_price, min_value


#Task two: Use the apply() method to create a new column directly in the DataFrame.

#Hint: In Step 4, we use a for loop to go through the option chain. Your task is to eliminate the for loop completely. Use Pandas vectorization to calculate the Mid Price (c), and then use the df.apply() method to calculate the implied volatility and store it as a new column in the calls DataFrame called Implied_Vol.

# Assuming Step 1, 2, and 3 have already run, and 'calls' is our dataframe.

# 1. Calculate Mid Price for the whole column at once 
calls['Mid_Price'] = (calls['Bid'] + calls['Ask']) / 2.0

# 2. Filter out bad data (where c <= 0)
calls_clean = calls[calls['Mid_Price'] > 0].copy()

# 3. Use lambda and apply to run the function across the dataframe
# axis=1 means "apply this function across each row"
calls_clean['Implied_Vol']=calls_clean.apply(lambda row: implied_vol_call_min(s, row['Strike'], t, r, row['Mid_Price']), axis=1)    

